# SECTION 1: IMPORTS & SETUP

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Tuple

pd.set_option("display.float_format", lambda x: f"{x:.4g}")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("All libraries imported.")

All libraries imported.


# SECTION 2: PHYSICAL CONSTANTS & ATMOSPHERE MODEL

In [ ]:
# --- Physical constants ---
R_AIR   = 287.05       # Specific gas constant for air [J/(kg·K)]
GAMMA   = 1.4          # Ratio of specific heats for air [-]
SIGMA   = 5.6704e-8    # Stefan-Boltzmann constant [W/(m²·K⁴)]
G0      = 9.80665      # Standard gravity [m/s²]
R_EARTH = 6.371e6      # Earth radius [m]

# --- Simplified exponential atmosphere layers ---
# Each layer: altitude base/top [m], sea-level density [kg/m³], scale height [m]
ATM_LAYERS = [
    {"h_base":     0, "h_top":  11000, "rho0": 1.2250,   "H": 8500},
    {"h_base": 11000, "h_top":  25000, "rho0": 0.3639,   "H": 6341},
    {"h_base": 25000, "h_top":  47000, "rho0": 0.08803,  "H": 7922},
    {"h_base": 47000, "h_top":  86000, "rho0": 0.001322, "H": 7257},
    {"h_base": 86000, "h_top": 200000, "rho0": 5.46e-6,  "H": 5877},
]

def atmosphere(h: float) -> Tuple[float, float, float]:
    h = max(h, 0.0)
    for layer in ATM_LAYERS:
        if h <= layer["h_top"]:
            rho = layer["rho0"] * np.exp(-(h - layer["h_base"]) / layer["H"])
            # Temperature: lapse rate below 11 km, isothermal above
            T = max(216.65, 288.15 - 0.0065 * min(h, 11000.0))
            P = rho * R_AIR * T
            return rho, P, T
    # Above all defined layers — near vacuum
    return 1e-10, 1e-10, 186.0

# Quick atmosphere sanity-check printout
print("\nAtmosphere check:")
for alt_km in [0, 10, 50, 80, 120]:
    rho, P, T = atmosphere(alt_km * 1000)
    print(f"  {alt_km:4d} km  rho={rho:.4e} kg/m3  P={P:.3e} Pa  T={T:.1f} K")


Atmosphere check:
     0 km  rho=1.2250e+00 kg/m3  P=1.013e+05 Pa  T=288.1 K
    10 km  rho=3.7775e-01 kg/m3  P=2.420e+04 Pa  T=223.1 K
    50 km  rho=8.7437e-04 kg/m3  P=5.438e+01 Pa  T=216.7 K
    80 km  rho=1.4007e-05 kg/m3  P=8.711e-01 Pa  T=216.7 K
   120 km  rho=1.6776e-08 kg/m3  P=1.043e-03 Pa  T=216.7 K


# SECTION 3: VEHICLE CONFIGURATION

In [ ]:
@dataclass
class Vehicle:
    name         : str   = "Generic Capsule"
    mass         : float = 8000.0
    nose_radius  : float = 1.5
    ref_area     : float = 7.07
    Cd           : float = 1.5
    emissivity   : float = 0.85
    wall_temp_ini: float = 300.0
    cp_tps       : float = 900.0    # J/(kg·K) — similar to AVCOAT ablator
    mpa_tps      : float = 50.0     # kg/m²

    beta: float = field(init=False)

    def __post_init__(self):
        # Ballistic coefficient — governs deceleration profile
        self.beta = self.mass / (self.Cd * self.ref_area)

    def summary(self):
        print(f"\n{'='*50}")
        print(f"  Vehicle   : {self.name}")
        print(f"  Mass      : {self.mass:.1f} kg")
        print(f"  Beta      : {self.beta:.2f} kg/m2  (ballistic coeff)")
        print(f"  Nose Rn   : {self.nose_radius:.2f} m")
        print(f"  Cd        : {self.Cd}")
        print(f"  Emissiv.  : {self.emissivity}")
        print(f"{'='*50}")

# Two vehicles to compare
capsule = Vehicle(
    name="Apollo-like Capsule",
    mass=5800.0, nose_radius=4.7, ref_area=12.0,
    Cd=1.5, emissivity=0.85, wall_temp_ini=300.0,
    cp_tps=900.0, mpa_tps=50.0
)
probe = Vehicle(
    name="Sharp-Nose Probe",
    mass=500.0, nose_radius=0.1, ref_area=0.05,
    Cd=1.0, emissivity=0.80, wall_temp_ini=300.0,
    cp_tps=750.0, mpa_tps=20.0
)
capsule.summary()
probe.summary()


  Vehicle   : Apollo-like Capsule
  Mass      : 5800.0 kg
  Beta      : 322.22 kg/m2  (ballistic coeff)
  Nose Rn   : 4.70 m
  Cd        : 1.5
  Emissiv.  : 0.85

  Vehicle   : Sharp-Nose Probe
  Mass      : 500.0 kg
  Beta      : 10000.00 kg/m2  (ballistic coeff)
  Nose Rn   : 0.10 m
  Cd        : 1.0
  Emissiv.  : 0.8


# SECTION 4: HEAT TRANSFER EQUATIONS

In [ ]:
def sutton_graves(rho: float, V: float, Rn: float,
                  k_sg: float = 1.7415e-4) -> float:
    return k_sg * np.sqrt(rho / Rn) * V**3


def radiative_cooling(T_wall: float, emissivity: float) -> float:
    return emissivity * SIGMA * T_wall**4


def update_wall_temp(q_conv: float, q_rad: float, T_wall: float,
                     dt: float, cp: float, mpa: float) -> float:
        dT_dt = (q_conv - q_rad) / (mpa * cp)
                return T_wall + dt * dT_dt


def dynamic_pressure(rho: float, V: float) -> float:
    return 0.5 * rho * V**2

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 13)

# SECTION 5: EQUATIONS OF MOTION

In [ ]:
def eom(h: float, V: float, vehicle: Vehicle, gamma: float) -> Tuple[float, float]:
    h = max(h, 0.0)
    V = max(V, 1.0)

    rho, _, _ = atmosphere(h)

    # Gravity corrected for altitude
    g = G0 * (R_EARTH / (R_EARTH + h))**2

    # Aerodynamic drag deceleration
    drag_accel = dynamic_pressure(rho, V) * vehicle.Cd * vehicle.ref_area / vehicle.mass

    dh_dt = V * np.sin(gamma)              # negative because sin(gamma) < 0
    dV_dt = -drag_accel - g * np.sin(gamma)

    return dh_dt, dV_dt

# SECTION 6: RK4 INTEGRATOR

In [ ]:
def rk4_step(h: float, V: float, dt: float,
             vehicle: Vehicle, gamma: float) -> Tuple[float, float]:
    def f(hi, Vi):
        return eom(hi, Vi, vehicle, gamma)

    dh1, dV1 = f(h, V)
    dh2, dV2 = f(h + 0.5*dt*dh1, V + 0.5*dt*dV1)
    dh3, dV3 = f(h + 0.5*dt*dh2, V + 0.5*dt*dV2)
    dh4, dV4 = f(h +     dt*dh3, V +     dt*dV3)

    h_new = h + (dt/6.0) * (dh1 + 2*dh2 + 2*dh3 + dh4)
    V_new = V + (dt/6.0) * (dV1 + 2*dV2 + 2*dV3 + dV4)
    return h_new, V_new

# SECTION 7: SIMULATION ENGINE

In [ ]:
def run_simulation(vehicle: Vehicle,
                   h0: float       = 120e3,
                   V0: float       = 7800.0,
                   gamma_deg: float = -6.5,
                   dt: float       = 2.0,
                   h_stop: float   = 5000.0,
                   t_max: float    = 1200.0) -> pd.DataFrame:

    gamma_rad = np.radians(gamma_deg)
    h, V      = float(h0), float(V0)
    T_wall    = vehicle.wall_temp_ini
    t         = 0.0
    records   = []
    q_history = []

    while h > h_stop and V > 200.0 and t < t_max:
        rho, _, T_atm = atmosphere(h)

        # Speed of sound (calorically perfect air)
        a_sound = np.sqrt(GAMMA * R_AIR * T_atm)
        mach    = V / a_sound

        # Heat fluxes
        q_conv = sutton_graves(rho, V, vehicle.nose_radius)
        q_rad  = radiative_cooling(T_wall, vehicle.emissivity)
        q_dyn  = dynamic_pressure(rho, V)

        # Update wall temperature (cap at 4000 K — ablator limit)
        T_wall = min(
            update_wall_temp(q_conv, q_rad, T_wall, dt,
                             vehicle.cp_tps, vehicle.mpa_tps),
            4000.0
        )
        T_wall = max(T_wall, 200.0)  # physical floor

        # Cumulative heat load [MJ/m²]
        q_history.append(q_conv)
        Q_total = np.trapezoid(q_history) * dt / 1e6

        records.append({
            "time [s]"          : round(t, 1),
            "altitude [km]"     : h / 1000.0,
            "velocity [m/s]"    : V,
            "mach [-]"          : mach,
            "density [kg/m3]"   : rho,
            "q_conv [W/m2]"     : q_conv,
            "q_rad [W/m2]"      : q_rad,
            "T_wall [K]"        : T_wall,
            "dyn_pressure [Pa]" : q_dyn,
            "heat_load [MJ/m2]" : Q_total,
        })

        # Advance state
        h, V = rk4_step(h, V, dt, vehicle, gamma_rad)
        t   += dt

    df = pd.DataFrame(records)
    print(f"  Done: {len(df)} steps | "
          f"Alt_final={df['altitude [km]'].iloc[-1]:.1f} km | "
          f"Peak_q={df['q_conv [W/m2]'].max():.3e} W/m2 | "
          f"Peak_T={df['T_wall [K]'].max():.0f} K | "
          f"Q_total={df['heat_load [MJ/m2]'].iloc[-1]:.2f} MJ/m2")
    return df

# SECTION 8: RUN BOTH VEHICLES

In [ ]:
print("\n--- Apollo-like Capsule ---")
df_cap = run_simulation(capsule, h0=120e3, V0=7800.0, gamma_deg=-6.5, dt=2.0)

print("--- Sharp-Nose Probe ---")
df_prb = run_simulation(probe,   h0=120e3, V0=7800.0, gamma_deg=-6.5, dt=2.0)

# SECTION 9: SUMMARY STATISTICS TABLE

In [ ]:
def mission_summary(df: pd.DataFrame, label: str) -> dict:
    return {
        "Vehicle"               : label,
        "Entry Duration [s]"    : df["time [s]"].iloc[-1],
        "Peak q_conv [MW/m2]"   : df["q_conv [W/m2]"].max()   / 1e6,
        "Peak T_wall [K]"       : df["T_wall [K]"].max(),
        "Peak Mach [-]"         : df["mach [-]"].max(),
        "Peak DynPres [kPa]"    : df["dyn_pressure [Pa]"].max() / 1e3,
        "Heat Load [MJ/m2]"     : df["heat_load [MJ/m2]"].iloc[-1],
    }

summary_df = pd.DataFrame([
    mission_summary(df_cap, "Apollo-like Capsule"),
    mission_summary(df_prb, "Sharp-Nose Probe"),
]).set_index("Vehicle")

print("\n========== MISSION SUMMARY TABLE ==========")
print(summary_df.to_string())

# SECTION 10: MAIN VISUALISATIONS (6-panel figure)

In [ ]:
fig = plt.figure(figsize=(16, 13))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.32)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[2, 1])

datasets = [
    (df_cap, "Capsule (blunt)", "steelblue"),
    (df_prb, "Sharp Probe",     "tomato"),
]

for df, lbl, col in datasets:
    t = df["time [s]"]

    # Plot 1: Altitude vs Time
    ax1.plot(t, df["altitude [km]"], color=col, lw=2, label=lbl)

    # Plot 2: Velocity vs Altitude (phase portrait)
    ax2.plot(df["velocity [m/s]"] / 1000, df["altitude [km]"],
             color=col, lw=2, label=lbl)

    # Plot 3: Heat Flux vs Time
    ax3.plot(t, df["q_conv [W/m2]"] / 1e6, color=col, lw=2, label=lbl)

    # Plot 4: Wall Temperature vs Time
    ax4.plot(t, df["T_wall [K]"], color=col, lw=2, label=lbl)

    # Plot 5: Integrated Heat Load
    ax5.plot(t, df["heat_load [MJ/m2]"], color=col, lw=2, label=lbl)

    # Plot 6: Mach Number vs Altitude
    ax6.plot(df["mach [-]"], df["altitude [km]"], color=col, lw=2, label=lbl)

# Annotations on wall temperature plot
ax4.axhline(1800, color="darkorange", ls="--", lw=1.3,
            label="TUFI tile limit ~1800 K")
ax4.axhline(3000, color="red",        ls="--", lw=1.3,
            label="Carbon limit ~3000 K")

# Mach reference lines
ax6.axvline(1.0,  color="gray",   ls=":",  lw=1.2, label="Mach 1")
ax6.axvline(5.0,  color="orange", ls="--", lw=1.0, label="Mach 5 (hypersonic)")

# Labels
for ax, xl, yl, title in [
    (ax1, "Time [s]",     "Altitude [km]",     "Altitude vs Time"),
    (ax2, "Velocity [km/s]", "Altitude [km]",  "Velocity–Altitude (Phase Space)"),
    (ax3, "Time [s]",     "Heat Flux [MW/m²]", "Stagnation Heat Flux"),
    (ax4, "Time [s]",     "Wall Temp [K]",     "TPS Wall Temperature"),
    (ax5, "Time [s]",     "Heat Load [MJ/m²]", "Cumulative Heat Load"),
    (ax6, "Mach [-]",     "Altitude [km]",     "Mach Number vs Altitude"),
]:
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)

fig.suptitle("Re-Entry Heating Analysis — Simplified Model",
             fontsize=15, fontweight="bold", y=1.01)
plt.savefig("reentry_main.png", bbox_inches="tight")
plt.show() # Display the plot in the notebook
plt.close()
print("\n\nMain figure saved: reentry_main.png")

# SECTION 11: ATMOSPHERE PROFILE PLOT

In [ ]:
alts = np.linspace(0, 120e3, 500)
rhos = [atmosphere(h)[0] for h in alts]
Ps   = [atmosphere(h)[1] for h in alts]
Ts   = [atmosphere(h)[2] for h in alts]

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5))
fig2.suptitle("Simplified Atmosphere Profile (0–120 km)", fontweight="bold")

axes2[0].plot(rhos, alts/1000, "steelblue", lw=2)
axes2[0].set(xlabel="Density [kg/m³]", ylabel="Altitude [km]",
             title="Density", xscale="log")

axes2[1].plot(Ps,   alts/1000, "seagreen",  lw=2)
axes2[1].set(xlabel="Pressure [Pa]", title="Pressure", xscale="log")

axes2[2].plot(Ts,   alts/1000, "tomato",    lw=2)
axes2[2].set(xlabel="Temperature [K]", title="Temperature")

for ax in axes2:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("atmosphere_profile.png", bbox_inches="tight")
plt.show() # Display the plot in the notebook
plt.close()
print("\n\nAtmosphere profile saved: atmosphere_profile.png")

# SECTION 12: PARAMETRIC STUDY — ENTRY ANGLE SWEEP

In [ ]:
angles  = [-2.0, -4.0, -6.5, -10.0, -15.0]
results = []
dfs_ang = {}

print("\nParametric sweep over entry angles (capsule):")
for ang in angles:
    df_a = run_simulation(capsule, h0=120e3, V0=7800.0,
                          gamma_deg=ang, dt=2.0, t_max=900.0)
    dfs_ang[ang] = df_a
    results.append({
        "Angle [deg]"          : ang,
        "Peak q [MW/m2]"       : df_a["q_conv [W/m2]"].max()  / 1e6,
        "Peak T_wall [K]"      : df_a["T_wall [K]"].max(),
        "Heat Load [MJ/m2]"    : df_a["heat_load [MJ/m2]"].iloc[-1],
        "Duration [s]"         : df_a["time [s]"].iloc[-1],
    })

param_df = pd.DataFrame(results)
print("\n===== PARAMETRIC STUDY — ENTRY ANGLE =====")
print(param_df.to_string(index=False))

# --- Parametric plots ---
fig3, axes3 = plt.subplots(1, 3, figsize=(16, 5))
fig3.suptitle("Parametric Study: Entry Angle Effect (Apollo Capsule)",
              fontweight="bold")

cmap   = plt.cm.coolwarm
colors = [cmap(i / max(len(angles)-1, 1)) for i in range(len(angles))]

for ang, col in zip(angles, colors):
    df_a  = dfs_ang[ang]
    label = f"{ang}°"
    axes3[0].plot(df_a["time [s]"], df_a["q_conv [W/m2]"]/1e6,
                  color=col, lw=1.8, label=label)
    axes3[1].plot(df_a["time [s]"], df_a["T_wall [K]"],
                  color=col, lw=1.8, label=label)
    axes3[2].plot(df_a["time [s]"], df_a["heat_load [MJ/m2]"],
                  color=col, lw=1.8, label=label)

for ax, yl, title in zip(
    axes3,
    ["Heat Flux [MW/m²]", "Wall Temp [K]", "Heat Load [MJ/m²]"],
    ["Peak Heat Flux vs Time", "Wall Temperature vs Time",
     "Cumulative Heat Load vs Time"]
):
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(yl)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
print("\n\n")
plt.tight_layout()
plt.savefig("parametric_angle.png", bbox_inches="tight")
plt.show() # Display the plot in the notebook
plt.close()
print("\n\nParametric figure saved: parametric_angle.png")

In [ ]:
angles_probe  = [-2.0, -4.0, -6.5, -10.0, -15.0] # Using the same angles for comparison
results_probe = []
dfs_ang_probe = {}

print("\nParametric sweep over entry angles (sharp-nose probe):")
for ang in angles_probe:
    # Run simulation for the probe vehicle
    df_p = run_simulation(probe, h0=120e3, V0=7800.0,
                          gamma_deg=ang, dt=2.0, t_max=900.0)
    dfs_ang_probe[ang] = df_p
    results_probe.append({
        "Angle [deg]"          : ang,
        "Peak q [MW/m2]"       : df_p["q_conv [W/m2]"].max()  / 1e6,
        "Peak T_wall [K]"      : df_p["T_wall [K]"].max(),
        "Heat Load [MJ/m2]"    : df_p["heat_load [MJ/m2]"].iloc[-1],
        "Duration [s]"         : df_p["time [s]"].iloc[-1],
    })

param_df_probe = pd.DataFrame(results_probe)
print("\n===== PARAMETRIC STUDY — ENTRY ANGLE (SHARP-NOSE PROBE) =====")
print(param_df_probe.to_string(index=False))

# --- Parametric plots for Sharp-Nose Probe ---
fig4, axes4 = plt.subplots(1, 3, figsize=(16, 5))
fig4.suptitle("Parametric Study: Entry Angle Effect (Sharp-Nose Probe)",
              fontweight="bold")

cmap_probe   = plt.cm.plasma # Using a different colormap for distinction
colors_probe = [cmap_probe(i / max(len(angles_probe)-1, 1)) for i in range(len(angles_probe))]

for ang, col in zip(angles_probe, colors_probe):
    df_p  = dfs_ang_probe[ang]
    label = f"{ang}°"
    axes4[0].plot(df_p["time [s]"], df_p["q_conv [W/m2]"]/1e6,
                  color=col, lw=1.8, label=label)
    axes4[1].plot(df_p["time [s]"], df_p["T_wall [K]"],
                  color=col, lw=1.8, label=label)
    axes4[2].plot(df_p["time [s]"], df_p["heat_load [MJ/m2]"],
                  color=col, lw=1.8, label=label)

for ax, yl, title in zip(
    axes4,
    ["Heat Flux [MW/m²]", "Wall Temp [K]", "Heat Load [MJ/m²]"],
    ["Peak Heat Flux vs Time", "Wall Temperature vs Time",
     "Cumulative Heat Load vs Time"]
):
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(yl)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
print("\n\n")
plt.tight_layout()
plt.savefig("parametric_angle_probe.png", bbox_inches="tight")
plt.show() # Display the plot in the notebook
plt.close()
print("\n\nParametric figure for probe saved: parametric_angle_probe.png")

# SECTION 13: SAMPLE DATA TABLE PRINTOUT

In [ ]:
print("\n===== CAPSULE SIMULATION — SAMPLE ROWS (every 20th) =====")
cols = ["time [s]", "altitude [km]", "velocity [m/s]",
        "mach [-]", "q_conv [W/m2]", "T_wall [K]", "heat_load [MJ/m2]"]
sample = df_cap.iloc[::max(1, len(df_cap)//20)].reset_index(drop=True)
print(sample[cols].to_string(index=False))

print("\n===== ALL SECTIONS COMPLETE =====")